# Análisis espectral de la aceleración — sistema biela-manivela
**Óptica y Ondas / UTEC 2026-II**

Cubre los puntos **3 (análisis espectral)** y **4 (segundo armónico y motor de 4 cilindros)**.

> ⚠️ **Tres correcciones respecto a la plantilla oficial**
> 1. **Columna equivocada.** La plantilla hace la FFT sobre `Aceleración2`, que en nuestros
>    archivos es la aceleración **angular** (rad/s²). Hay que usar la **lineal** (m/s²).
> 2. **Bug de fase.** La plantilla reconstruye con `np.cos(2*np.pi*f*t + phi)` usando el
>    tiempo **absoluto**, pero las fases de la FFT están referidas a la **primera muestra**
>    de la ventana. Si la ventana empieza en t = 1 s, la reconstrucción sale desfasada.
>    Aquí se usa `tau = t - t[0]`.
> 3. **Ventana arbitraria.** Elegir el intervalo "a ojo" mete fuga espectral. Aquí la ventana
>    se cierra en un número **entero de vueltas** usando la columna de ángulo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

COLUMNAS = ["t", "x", "v", "a", "theta", "omega", "alpha"]
#            s   m  m/s  m/s²   rad     rad/s    rad/s²

CORRIDAS = {
    "10Hz-c1": "biela-manivela-10Hz-08-09-2026-c1.txt",
    "10Hz-c2": "biela-manivela-10Hz-08-09-2026-c2.txt",
    "20Hz-c1": "biela-manivela-20Hz-08-09-2026-c1.txt",
    "20Hz-c2": "biela-manivela-20Hz-08-09-2026-c2.txt",
    "30Hz-c1": "biela-manivela-30Hz-08-09-2026-c1.txt",   # DESCARTADA: ver data/NOTAS.md
    "30Hz-c2": "biela-manivela-30Hz-08-09-2026-c2.txt",
}

def dir_datos():
    for c in [Path("../data/raw"), Path("data/raw"), Path(".")]:
        if (c / CORRIDAS["10Hz-c2"]).exists():
            return c
    raise FileNotFoundError("No encuentro data/raw. Ejecuta el notebook desde "
                            "notebooks/ o desde la raiz del repo.")

DIR_FIG = Path("../figuras") if Path("../figuras").exists() else Path("figuras")
DIR_FIG.mkdir(exist_ok=True)

def guardar(nombre):
    p = DIR_FIG / f"{nombre}.png"
    plt.savefig(p, dpi=150, bbox_inches="tight")
    print(f"figura guardada: {p}")

def cargar(corrida):
    d = np.genfromtxt(dir_datos() / CORRIDAS[corrida],
                      skip_header=7, encoding="utf-8-sig")
    d = d[~np.isnan(d).any(axis=1)]
    return {c: d[:, i] for i, c in enumerate(COLUMNAS)}

def R2(y, y_fit):
    return 1 - np.sum((y - y_fit)**2) / np.sum((y - y.mean())**2)

## 1. Cargar y acotar a un número entero de vueltas

En vez de elegir el intervalo a ojo, usamos la columna de ángulo para cortar exactamente
donde se completa la última vuelta entera. Así la señal empieza y termina en la misma
fase y la fuga espectral se reduce al mínimo.

In [ ]:
CORRIDA = "10Hz-c2"       # 10Hz-c1 | 10Hz-c2 | 20Hz-c1 | 20Hz-c2 | 30Hz-c2
T_INICIO = {"10Hz-c1": 1.0, "10Hz-c2": 1.0, "20Hz-c1": 1.0,
            "20Hz-c2": 1.0, "30Hz-c2": 1.2}
# Tope superior: 20Hz-c2 pierde el carrito despues de t=11.2 s (la ultima
# vuelta registra 13 cm de recorrido en vez de 21). None = usar todo.
T_FIN = {"10Hz-c1": None, "10Hz-c2": None, "20Hz-c1": None,
         "20Hz-c2": 11.1, "30Hz-c2": None}
if CORRIDA not in T_INICIO:
    raise ValueError(f"{CORRIDA} no es analizable (ver data/NOTAS.md). "
                     f"Usa una de: {list(T_INICIO)}")
T_INICIO, T_FIN = T_INICIO[CORRIDA], T_FIN[CORRIDA]
L_BIELA = 0.36

D = cargar(CORRIDA)
i0 = int(np.searchsorted(D["t"], T_INICIO))
iF = len(D["t"]) if T_FIN is None else int(np.searchsorted(D["t"], T_FIN))

# ángulo recorrido desde el inicio, monótono creciente
th_rel = np.abs(D["theta"][:iF] - D["theta"][i0])
n_vueltas = int(th_rel[-1] // (2*np.pi))
i1 = int(np.argmin(np.abs(th_rel - n_vueltas*2*np.pi)))

t   = D["t"][i0:i1+1]
x   = D["x"][i0:i1+1]
a   = D["a"][i0:i1+1]          # aceleración LINEAL
th  = D["theta"][i0:i1+1]
om  = D["omega"][i0:i1+1]
al  = D["alpha"][i0:i1+1]      # aceleración ANGULAR

tau = t - t[0]                 # tiempo relativo: imprescindible para las fases
N   = len(t)
dt  = np.median(np.diff(t))
fs  = 1/dt
T_total = N*dt
df  = 1/T_total

print(f"{CORRIDA}")
print(f"  ventana        : [{t[0]:.2f}, {t[-1]:.2f}] s  ->  {n_vueltas} vueltas completas")
print(f"  N muestras     : {N}      fs = {fs:.1f} Hz")
print(f"  T_total        : {T_total:.3f} s")
print(f"  resolución Δf  : {df:.4f} Hz")
print(f"  f1 esperada    : {n_vueltas/T_total:.4f} Hz  (= {n_vueltas} vueltas / T_total)")

## 2. Punto 3 — DFT de la aceleración lineal y espectro de amplitudes

In [ ]:
a_dc = a.mean()
a_ac = a - a_dc                       # quitar el nivel continuo antes de transformar

Y    = np.fft.rfft(a_ac)
frec = np.fft.rfftfreq(N, dt)
amp  = 2/N * np.abs(Y)
fase = np.angle(Y)

# Primer armónico: el pico global (excluyendo el bin de continua)
k1 = 1 + int(np.argmax(amp[1:]))
f1, A1, phi1 = frec[k1], amp[k1], fase[k1]

# Segundo armónico: el pico mayor en la banda alrededor de 2*f1
banda = (frec > 1.5*f1) & (frec < 2.5*f1)
k2 = int(np.where(banda)[0][np.argmax(amp[banda])])
f2, A2, phi2 = frec[k2], amp[k2], fase[k2]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.stem(frec, amp, basefmt=" ", markerfmt="o", linefmt="-")
ax.plot(f1, A1, "o", ms=13, mfc="none", mec="crimson", mew=2.2,
        label=f"1er armónico: {f1:.3f} Hz, A₁ = {A1:.4f}")
ax.plot(f2, A2, "o", ms=13, mfc="none", mec="tab:green", mew=2.2,
        label=f"2do armónico: {f2:.3f} Hz, A₂ = {A2:.4f}")
ax.axvline(2*f1, ls=":", c="tab:green", lw=1.5, label=f"2·f₁ = {2*f1:.3f} Hz (teórico)")
ax.set_xlim(0, min(6, fs/2)); ax.set_xlabel("Frecuencia (Hz)")
ax.set_ylabel("Amplitud (m/s²)"); ax.legend()
ax.set_title(f"Espectro de amplitudes de a(t) — {CORRIDA}")
plt.tight_layout(); guardar(f"05_espectro_{CORRIDA}"); plt.show()

print(f"f₁ = {f1:.4f} Hz    A₁ = {A1:.4f} m/s²    φ₁ = {phi1:+.4f} rad")
print(f"f₂ = {f2:.4f} Hz    A₂ = {A2:.4f} m/s²    φ₂ = {phi2:+.4f} rad")
print(f"f₂/f₁ = {f2/f1:.3f}     (teórico exacto: 2.000)")
print(f"A₂/A₁ = {A2/A1:.3f}     (teórico r/L = {A1/(2*np.pi*f1)**2/L_BIELA:.3f})")
print(f"Δf = {df:.4f} Hz  ->  la diferencia entre f₂ y 2·f₁ es de "
      f"{abs(f2-2*f1)/df:.1f} bines del espectro")

## 3. Punto 3 — Reconstrucción con los dos primeros armónicos

$a_{\rm aprox}(t) = A_1\cos(2\pi f_1 \tau + \phi_1) + A_2\cos(2\pi f_2 \tau + \phi_2)$

con $\tau = t - t_0$. **Esta es la corrección de fase**: usar `t` absoluto desfasa todo.

In [ ]:
def armonico(tau, A, f, phi):
    return A*np.cos(2*np.pi*f*tau + phi)

rec_1 = a_dc + armonico(tau, A1, f1, phi1)
rec_2 = rec_1 + armonico(tau, A2, f2, phi2)

fig, ax = plt.subplots(2, 1, figsize=(11, 7.5), sharex=True,
                       gridspec_kw={"height_ratios": [3, 1.5]})

ax[0].plot(t, a, "o", ms=3, alpha=.5, label="a(t) medida")
ax[0].plot(t, rec_1, "--", lw=1.6, color="crimson",
           label=f"solo 1er armónico (R² = {R2(a, rec_1):.3f})")
ax[0].plot(t, rec_2, "-", lw=2.2, color="tab:blue",
           label=f"1er + 2do armónico (R² = {R2(a, rec_2):.3f})")
ax[0].set_ylabel("Aceleración (m/s²)"); ax[0].legend(loc="upper right", fontsize=9)
ax[0].set_title(f"Señal original vs reconstruida — {CORRIDA}")

ax[1].plot(t, a - rec_1, "o", ms=3, alpha=.5, color="crimson",
           label=f"residuo 1 armónico (rms {np.std(a-rec_1):.4f})")
ax[1].plot(t, a - rec_2, "o", ms=3, alpha=.6, color="tab:blue",
           label=f"residuo 2 armónicos (rms {np.std(a-rec_2):.4f})")
ax[1].axhline(0, c="k", lw=.8); ax[1].legend(fontsize=8)
ax[1].set_ylabel("Residuo (m/s²)"); ax[1].set_xlabel("Tiempo (s)")
plt.tight_layout(); guardar(f"06_reconstruccion_{CORRIDA}"); plt.show()

print(f"R² con 1 armónico  : {R2(a, rec_1):.4f}")
print(f"R² con 2 armónicos : {R2(a, rec_2):.4f}")
print(f"Mejora en varianza del residuo: "
      f"{100*(1 - np.var(a-rec_2)/np.var(a-rec_1)):.1f} %")

**Qué conserva y qué pierde la reconstrucción** (para comentar en el video)

- **Conserva:** el periodo, la amplitud global y la asimetría entre los dos extremos del
  recorrido (picos y valles de distinta forma). Eso es justamente lo que el MAS puro no puede.
- **Pierde:** el ruido de alta frecuencia, las variaciones de amplitud entre ciclos
  (consecuencia de que la mano no gira parejo) y los armónicos de orden 3 en adelante.

## 4. ¿Por qué f₂/f₁ no sale exactamente 2 y A₂/A₁ es inestable?

Esto **no es un error del análisis**: es física del montaje, y es el punto más valioso
para discutir.

La teoría de la guía (ec. 8) supone giro uniforme, θ = ωt. Si ω **no** es constante, la
regla de la cadena da un término extra:

$$x = f(\theta) \;\Rightarrow\; v = f'(\theta)\,\omega \;\Rightarrow\;
a = \underbrace{f''(\theta)\,\omega^2}_{\text{armónicos geométricos}} +
    \underbrace{f'(\theta)\,\alpha}_{\text{extra por giro no uniforme}}$$

El segundo término solo existe porque giramos a mano. La celda siguiente mide su tamaño.

In [ ]:
# Ajuste geométrico de x(θ) con dos armónicos
M   = np.c_[np.ones_like(th), np.cos(th), np.sin(th), np.cos(2*th), np.sin(2*th)]
c, *_ = np.linalg.lstsq(M, x, rcond=None)

dM  = np.c_[np.zeros_like(th), -np.sin(th),   np.cos(th),
            -2*np.sin(2*th),    2*np.cos(2*th)]
d2M = np.c_[np.zeros_like(th), -np.cos(th),  -np.sin(th),
            -4*np.cos(2*th),   -4*np.sin(2*th)]

fp, fpp = dM @ c, d2M @ c            # f'(θ) y f''(θ)

term_geo   = fpp * om**2             # armónicos geométricos
term_alpha = fp  * al                # contaminación por α
a_modelo   = term_geo + term_alpha

A_geo   = np.hypot(c[1], c[2])
B_geo   = np.hypot(c[3], c[4])
amp_2do = 4 * B_geo * (om**2).mean()          # amplitud del 2do armónico geométrico
amp_alf = np.std(term_alpha) * np.sqrt(2)     # amplitud equivalente del término de α

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(t, a, "o", ms=3, alpha=.45, label="a(t) medida")
ax.plot(t, a_modelo, "-", lw=1.8, color="tab:blue",
        label=f"f''(θ)ω² + f'(θ)α   (R² = {R2(a, a_modelo):.3f})")
ax.plot(t, term_alpha, "-", lw=1.5, color="darkorange",
        label="solo el término f'(θ)·α")
ax.axhline(0, c="k", lw=.7)
ax.set_xlabel("Tiempo (s)"); ax.set_ylabel("Aceleración (m/s²)"); ax.legend(fontsize=9)
ax.set_title("La aceleración medida no es solo geométrica")
plt.tight_layout(); guardar(f"07_termino_alpha_{CORRIDA}"); plt.show()

print(f"Amplitud del 2do armónico geométrico : {amp_2do:.4f} m/s²")
print(f"Amplitud del término de α            : {amp_alf:.4f} m/s²")
print(f"Razón contaminación / señal          : {amp_alf/amp_2do:.2f}")
print()
print("Conclusión: el término parásito es del MISMO ORDEN que el segundo armónico.")
print("Por eso el pico del espectro se corre de 2·f1 y su amplitud no es fiable.")

## 5. Ruta robusta — DFT en el dominio del ángulo

La posición x(θ) es **puramente geométrica**: no depende de ω ni de α. Si hacemos la DFT
contra el ángulo en lugar del tiempo, el eje queda en **ciclos por vuelta** y los armónicos
caen en n = 1 y n = 2 exactos, sin importar cómo giró la mano.

Es exactamente el eje de la **figura 8** de la guía ("Armónico (ciclos por vuelta)").

In [ ]:
MUESTRAS_POR_VUELTA = 64

th_u = np.abs(th - th[0])                                  # 0 .. n_vueltas*2π
rejilla = np.linspace(0, n_vueltas*2*np.pi,
                      n_vueltas*MUESTRAS_POR_VUELTA, endpoint=False)
x_ang = np.interp(rejilla, th_u, x)
x_ang = x_ang - x_ang.mean()

Mg   = len(x_ang)
Yg   = np.fft.rfft(x_ang)
orden = np.fft.rfftfreq(Mg, 1/MUESTRAS_POR_VUELTA)         # ciclos por vuelta
ampg = 2/Mg * np.abs(Yg)

n1 = int(np.argmin(np.abs(orden - 1)))
n2 = int(np.argmin(np.abs(orden - 2)))

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.stem(orden, 100*ampg, basefmt=" ")
ax.plot(orden[n1], 100*ampg[n1], "o", ms=13, mfc="none", mec="crimson", mew=2.2,
        label=f"n = 1 :  {100*ampg[n1]:.2f} cm   (= r)")
ax.plot(orden[n2], 100*ampg[n2], "o", ms=13, mfc="none", mec="tab:green", mew=2.2,
        label=f"n = 2 :  {100*ampg[n2]:.3f} cm   (= r²/4L)")
ax.set_xlim(0, 10); ax.set_xlabel("Armónico (ciclos por vuelta)")
ax.set_ylabel("Amplitud de x (cm)"); ax.legend()
ax.set_title(f"DFT de la posición en el dominio del ángulo — {CORRIDA}")
plt.tight_layout(); guardar(f"08_dft_angular_{CORRIDA}"); plt.show()

r_dft  = ampg[n1]
B_dft  = ampg[n2]
print(f"n = 1  ->  r      = {100*r_dft:.2f} cm")
print(f"n = 2  ->  r²/4L  = {100*B_dft:.3f} cm     teórico = {100*r_dft**2/(4*L_BIELA):.3f} cm")
print(f"Razón A₂/A₁ en posición    : {B_dft/r_dft:.4f}   (teórico r/4L = {r_dft/(4*L_BIELA):.4f})")
print(f"Razón A₂/A₁ en aceleración : {4*B_dft/r_dft:.4f}   (teórico r/L  = {r_dft/L_BIELA:.4f})")
print()
print("Al derivar dos veces, cos(2θ) se multiplica por 4: de ahí el factor 4.")
print("Por eso la anarmonicidad se nota mucho más en la aceleración que en la posición.")

## 6. Punto 4 — Discusión: fuente de los armónicos y el motor de 4 cilindros

**¿De dónde sale cada armónico?**

- **Primer armónico (n = 1, f₁).** Es la proyección del giro de la manivela sobre el eje del
  riel: el término r·cos θ. Existiría igual con una biela infinitamente larga. Es el MAS.
- **Segundo armónico (n = 2, 2f₁).** Es puramente geométrico, de la **longitud finita de la
  biela**. Al inclinarse, la biela no transmite el giro de forma simétrica: el pistón pasa
  más rápido por un extremo que por el otro. Su amplitud es r²/4L en posición y r²ω²/L en
  aceleración — proporcional a r/L, así que desaparece si L → ∞.
- **Armónicos superiores y picos no enteros.** En nuestro caso vienen de tres fuentes
  identificadas: giro no uniforme (el término f'(θ)·α medido arriba), ruido de
  cuantización amplificado por la doble derivada numérica, y juego mecánico en los pernos
  más fricción carrito-riel.

**Implicancias en un motor de 4 cilindros en línea**

- El cigüeñal reparte los pistones a 180° entre sí. Eso hace que los **primeros armónicos
  se cancelen por pares**: dos pistones suben mientras dos bajan, y las fuerzas de inercia
  de frecuencia f se compensan.
- Pero el segundo armónico va al **doble** de frecuencia: al desfasar 180° en el giro,
  el término cos(2θ) se desfasa 360°, o sea **queda en fase**. En vez de cancelarse, los
  cuatro pistones **suman**. La fuerza vertical residual es ≈ 4·m·ω²·(r²/L).
- Consecuencia práctica: el motor de 4 cilindros en línea vibra a **2× las revoluciones**.
  Por eso los diseños que buscan suavidad usan **ejes de balance contrarrotantes al doble
  de giro** (Lanchester), o pasan a arquitecturas donde el segundo armónico sí se cancela
  (V6 a 60°, bóxer de 6, I6).
- Y con nuestros números: A₂/A₁ ≈ r/L. Con r/L ≈ 0.3 el segundo armónico vale ~30% del
  primero en un solo pistón, pero al no cancelarse en el conjunto termina siendo la
  **fuente dominante** de vibración del motor completo. Ahí está la razón de que el
  laboratorio insista en medirlo.